# Day 1 AM — Agentic Operations Foundations

Run this notebook top-to-bottom with a live Anthropic model. Domain facts come only from the insurance fixtures under `Week2/data/insurance`.

The study-only app at `../app/README.md` shows package structure; learners execute **this** notebook.

Two **Exercises** appear near the end (after §06). Attempt them before opening [`_EXERCISES_SOLUTIONS.ipynb`](_EXERCISES_SOLUTIONS.ipynb).

Later cells may redefine names. Restart the kernel before jumping backward mid-notebook.

<img src="Images/d1am_non_agentic_vs_agentic_workflows.png" width="900" alt="Non-agentic linear workflow vs application-controlled agentic loop">

- **Non-agentic AI** — one prompt, one response; best when context and steps are already known.
- **Agentic AI** — model inside an application-owned loop: perceive → reason → act → observe.
- **Use a pipeline** when the path is fixed; **use an agent** when the next tool depends on observations.
- The application owns tools, state, evidence checks, budgets, and stop conditions.

In [1]:
print()

In [2]:
from __future__ import annotations

import csv
import json
import os
import sqlite3
import time
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from IPython.display import JSON, display
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field


def discover_week2_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        for possible in (candidate, candidate / "Week2"):
            if (possible / "requirements.txt").is_file() and (
                possible / "data" / "insurance" / "fnol_emails.csv"
            ).is_file():
                return possible
    raise FileNotFoundError("Could not locate Week2. Start Jupyter from the repo or Week2.")


WEEK2_ROOT = discover_week2_root()
FNOL_PATH = WEEK2_ROOT / "data" / "insurance" / "fnol_emails.csv"
CLAIMS_DB = WEEK2_ROOT / "data" / "insurance" / "claims.db"
HANDOFF_DIR = WEEK2_ROOT / "W2D1" / "outputs" / "day1"
load_dotenv(WEEK2_ROOT / ".env")


def make_model(*, max_tokens: int | None = None) -> ChatAnthropic:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set ANTHROPIC_API_KEY in Week2/.env before running Day 1.")
    return ChatAnthropic(
        model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
        max_tokens=max_tokens or int(os.getenv("AGENT_MAX_TOKENS", "1200")),
        temperature=0,
        max_retries=1,
    )


TURN_BUDGET = int(os.getenv("AGENT_MAX_STEPS", "8"))
TOKEN_BUDGET = int(os.getenv("AGENT_MAX_TOKENS", "1200"))

if not FNOL_PATH.is_file():
    raise FileNotFoundError(f"Missing FNOL corpus: {FNOL_PATH}")
if not CLAIMS_DB.is_file():
    raise FileNotFoundError(
        f"Missing {CLAIMS_DB}. From Week2 root run: "
        "uv run python data/insurance/seed_claims_db.py"
    )

display(
    JSON(
        {
            "week2_root": str(WEEK2_ROOT),
            "model": os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
            "turn_budget": TURN_BUDGET,
            "token_budget": TOKEN_BUDGET,
        }
    )
)

c:\Users\Asus\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<IPython.core.display.JSON object>

## 01 The agentic loop

**Perceive → reason → act → observe.** The LLM plans and requests tools; the application owns state, tool execution, and stop rules. Hidden chain-of-thought is not an audit trail—keep typed decisions, tool calls, observations, and stop reasons.

<img src="Images/d1am_t1_agent_loop.png" width="850" alt="Perceive-reason-act-observe agent loop">

<img src="Images/d1am_t1_agent_graph.png" width="850" alt="Controlled execution graph with validate, authorize, budget, and execute">

Demo path: typed `RoutingDecision`, then a live `MessagesState` + `ToolNode` FNOL lookup loop on `CLN-001`.

In [3]:
class RoutingDecision(BaseModel):
    workflow: Literal["agent", "pipeline", "hybrid"]
    reason: str
    requires_domain_tool: bool


routing = make_model().with_structured_output(RoutingDecision).invoke(
    "Operator request: triage FNOL email CLN-001, verify claim and policy records, "
    "then recommend a queue. Choose agent, pipeline, or hybrid. Facts must come from tools."
)
display(JSON(routing.model_dump()))

<IPython.core.display.JSON object>

In [4]:
@tool
def fnol_lookup(email_id: str) -> str:
    """Return one authoritative FNOL email row by CLN id, or found:false."""
    with FNOL_PATH.open(encoding="utf-8", newline="") as handle:
        row = next(
            (item for item in csv.DictReader(handle) if item["email_id"] == email_id),
            None,
        )
    return json.dumps(row or {"found": False, "email_id": email_id})


def build_fnol_loop():
    model = make_model().bind_tools([fnol_lookup])

    def call_model(state: MessagesState):
        return {"messages": [model.invoke(state["messages"])]}

    def route(state: MessagesState):
        return "tools" if state["messages"][-1].tool_calls else END

    builder = StateGraph(MessagesState)
    builder.add_node("agent", call_model)
    builder.add_node("tools", ToolNode([fnol_lookup]))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()


fnol_graph = build_fnol_loop()
print("FNOL ToolNode loop compiled.")

FNOL ToolNode loop compiled.


In [5]:
fnol_result = fnol_graph.invoke(
    {
        "messages": [
            SystemMessage(
                content=(
                    "Call fnol_lookup for facts. Never invent a record. "
                    "After the tool result, give a concise operator summary."
                )
            ),
            HumanMessage(content="Triage FNOL email CLN-001."),
        ]
    },
    config={"recursion_limit": TURN_BUDGET},
)

for message in fnol_result["messages"]:
    if isinstance(message, AIMessage):
        display(
            JSON(
                {
                    "tool_calls": message.tool_calls,
                    "usage": message.usage_metadata,
                    "stop_reason": message.response_metadata.get("stop_reason"),
                }
            )
        )
display(fnol_result["messages"][-1])

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

AIMessage(content='Here is the triage summary for **CLN-001**:\n\n---\n\n### 📋 FNOL Triage Summary — CLN-001\n\n| Field | Details |\n|---|---|\n| **Email ID** | CLN-001 |\n| **Subject** | Property claim CLM-424063 — fence damage |\n| **Claimant** | Jennifer Garcia |\n| **Claim Number** | CLM-424063 |\n| **Policy ID** | POL-787532354 |\n| **Loss Location** | 6650 Broadway |\n| **Loss Type** | Vandalism / Property Damage (fence) |\n| **Estimated Damage** | $5,834 |\n| **Police Report** | PR-20250822-4864 |\n| **Supporting Docs** | Photos available |\n\n---\n\n### ⚡ Triage Decision\n\n| Attribute | Assessment |\n|---|---|\n| **Urgency** | 🔴 **High** |\n| **Routing Queue** | `fnol_intake` |\n| **Category** | FNOL |\n\n---\n\n### 📝 Operator Notes\n- **Immediate action required** — vandalism claim with a police report filed, indicating a confirmed incident.\n- Estimated damage of **$5,834** warrants prompt review and adjuster assignment.\n- **Retrieve photos** from claimant to support damage

In [6]:
class BoundedState(MessagesState):
    model_calls: int
    stop_reason: str


def build_bounded_fnol(max_model_calls: int):
    model = make_model().bind_tools([fnol_lookup])

    def call_model(state: BoundedState):
        calls = state.get("model_calls", 0)
        if calls >= max_model_calls:
            return {"stop_reason": "model_call_budget_exhausted"}
        response = model.invoke(state["messages"])
        return {
            "messages": [response],
            "model_calls": calls + 1,
            "stop_reason": "tool_requested" if response.tool_calls else "completed",
        }

    def route(state: BoundedState):
        if state.get("stop_reason") == "model_call_budget_exhausted":
            return END
        return "tools" if state["messages"][-1].tool_calls else END

    builder = StateGraph(BoundedState)
    builder.add_node("agent", call_model)
    builder.add_node("tools", ToolNode([fnol_lookup], handle_tool_errors=True))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()


bounded = build_bounded_fnol(max_model_calls=2).invoke(
    {
        "messages": [
            SystemMessage(content="Call fnol_lookup before answering."),
            HumanMessage(content="Triage FNOL email CLN-001."),
        ],
        "model_calls": 0,
        "stop_reason": "not_started",
    },
    config={"recursion_limit": TURN_BUDGET},
)
display(JSON({"model_calls": bounded["model_calls"], "stop_reason": bounded["stop_reason"]}))

<IPython.core.display.JSON object>

## 02 Reasoning & planning patterns

<img src="Images/d1am_t2_reasoning_patterns.png" width="850" alt="ReAct, plan-and-execute, reflection, tree-of-thought">

- **ReAct (primary live demo)** — the Topic 01 tool loop: reason, act with a tool, observe, repeat.
- **Plan-and-execute** — emit a short structured plan, then execute steps.
- **Reflection / self-critique** — model critiques a draft `TriageDecision`.
- **Tree-of-thought** — compare a few bounded branches before choosing.
- Do not treat hidden CoT as the operator audit trail.

In [7]:
class Plan(BaseModel):
    goal: str
    steps: list[str] = Field(min_length=2, max_length=5)


plan = make_model().with_structured_output(Plan).invoke(
    "Create a short evidence-first plan to triage CLN-001 using fnol_lookup, "
    "then claim and policy tools only if identifiers appear in FNOL results."
)
display(JSON(plan.model_dump()))

# Execute the first planned step against the authoritative FNOL fixture.
step1 = json.loads(fnol_lookup.invoke({"email_id": "CLN-001"}))
display(JSON({"executed_step": plan.steps[0], "fnol_result": step1}))

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [8]:
class TriageDecision(BaseModel):
    lane: Literal["insurance", "banking", "logistics"]
    case_id: str
    urgency: Literal["low", "medium", "high"]
    route_queue: str
    summary: str
    rationale: str
    tools_used: list[str]
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_approval: bool = False


class Critique(BaseModel):
    unsupported_claims: list[str]
    missing_evidence: list[str]
    revised_summary: str


draft = TriageDecision(
    lane="insurance",
    case_id="CLM-424063",
    urgency="high",
    route_queue="fnol_intake",
    summary="Fence damage FNOL for Jennifer Garcia; route to intake.",
    rationale="Assumes coverage without re-checking policy status.",
    tools_used=["fnol_lookup"],
    confidence=0.55,
    needs_human_approval=True,
)

critique = make_model().with_structured_output(Critique).invoke(
    "Critique this draft TriageDecision using only the FNOL row. "
    "List unsupported claims and missing evidence; revise the summary.\n"
    f"Draft: {draft.model_dump_json()}\nFNOL: {json.dumps(step1)}"
)
display(JSON({"draft": draft.model_dump(), "critique": critique.model_dump()}))

<IPython.core.display.JSON object>

In [9]:
class BranchScore(BaseModel):
    branch: str
    score: float = Field(ge=0.0, le=1.0)
    why: str


class BranchCompare(BaseModel):
    branches: list[BranchScore] = Field(min_length=2, max_length=3)
    selected: str
    reason: str


tot = make_model().with_structured_output(BranchCompare).invoke(
    "Compare these bounded routes for CLN-001 using only the FNOL row. "
    "Branches: A=fnol_intake, B=policy_servicing, C=special_investigations. "
    f"FNOL: {json.dumps(step1)}"
)
display(JSON(tot.model_dump()))

<IPython.core.display.JSON object>

### Reasoning / thinking models (2026)

Some models expose extended thinking or reasoning modes. Treat that as optional model behavior, not as the compliance record. Persist the **plan**, **tool calls**, **tool results**, **decision object**, and **stop_reason** instead of relying on hidden CoT.

## 03 Tools as contracts

<img src="Images/d1am_t3_tool_contracts.png" width="850" alt="Typed tool contracts feeding TriageDecision">

Tools expose JSON Schema via Pydantic `args_schema`. Invoke with `tool_choice` when you need a forced tool turn. A valid `TriageDecision` shape is not enough—ground it in captured evidence. Budgets (`TURN_BUDGET`, `TOKEN_BUDGET`) stay visible in settings.

In [10]:
class FnolInput(BaseModel):
    email_id: str = Field(pattern=r"^CLN-\d{3}$")


class ClaimInput(BaseModel):
    claim_id: str = Field(pattern=r"^CLM-\d{6}$")


class PolicyInput(BaseModel):
    policy_id: str = Field(pattern=r"^POL-\d+$")


def db_row(table: str, column: str, value: str) -> str:
    with sqlite3.connect(CLAIMS_DB) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(f"SELECT * FROM {table} WHERE {column} = ?", (value,)).fetchone()
    return json.dumps(dict(row) if row else {"found": False, column: value})


@tool(args_schema=FnolInput)
def fnol_lookup(email_id: str) -> str:
    """Return one FNOL fixture row, or found:false."""
    with FNOL_PATH.open(encoding="utf-8", newline="") as handle:
        row = next(
            (item for item in csv.DictReader(handle) if item["email_id"] == email_id),
            None,
        )
    return json.dumps(row or {"found": False, "email_id": email_id})


@tool(args_schema=ClaimInput)
def claim_lookup(claim_id: str) -> str:
    """Return one claim row from claims.db, or found:false."""
    return db_row("claims", "claim_id", claim_id)


@tool(args_schema=PolicyInput)
def policy_lookup(policy_id: str) -> str:
    """Return one policy row from claims.db, or found:false."""
    return db_row("policies", "policy_id", policy_id)


TOOLS = [fnol_lookup, claim_lookup, policy_lookup]
display(
    JSON(
        {
            "tools": [t.name for t in TOOLS],
            "fnol_schema": FnolInput.model_json_schema(),
            "turn_budget": TURN_BUDGET,
            "token_budget": TOKEN_BUDGET,
        }
    )
)

<IPython.core.display.JSON object>

In [11]:
forced = make_model().bind_tools(TOOLS, tool_choice="any").invoke(
    [
        SystemMessage(content="Select exactly one domain tool. Do not answer from memory."),
        HumanMessage(content="Retrieve FNOL email CLN-001."),
    ]
)
call = forced.tool_calls[0]
observed = TOOLS[[t.name for t in TOOLS].index(call["name"])].invoke(call["args"])
display(
    JSON(
        {
            "tool_call": call,
            "tool_result": json.loads(observed),
            "stop_reason": forced.response_metadata.get("stop_reason"),
        }
    )
)

<IPython.core.display.JSON object>

In [12]:
def build_grounded_graph():
    model = make_model().bind_tools(TOOLS)

    def call_model(state: MessagesState):
        return {"messages": [model.invoke(state["messages"])]}

    def route(state: MessagesState):
        return "tools" if state["messages"][-1].tool_calls else END

    builder = StateGraph(MessagesState)
    builder.add_node("agent", call_model)
    builder.add_node("tools", ToolNode(TOOLS, handle_tool_errors=True))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()


def collect_evidence(email_id: str) -> list[dict]:
    result = build_grounded_graph().invoke(
        {
            "messages": [
                SystemMessage(
                    content=(
                        "Call fnol_lookup first. Then call claim_lookup and policy_lookup "
                        "only for identifiers present in the FNOL result. Never invent IDs."
                    )
                ),
                HumanMessage(content=f"Collect triage evidence for {email_id}."),
            ]
        },
        config={"recursion_limit": TURN_BUDGET},
    )
    return [
        {"tool": m.name, "result": json.loads(str(m.content))}
        for m in result["messages"]
        if isinstance(m, ToolMessage)
    ]


def validate_grounding(decision: TriageDecision, evidence: list[dict]) -> None:
    blob = json.dumps(evidence)
    if not decision.case_id.startswith("CLM-") or decision.case_id not in blob:
        raise ValueError("case_id is not an evidence-backed claim id")
    if not set(decision.tools_used).issubset({e["tool"] for e in evidence}):
        raise ValueError("tools_used lists a tool without evidence")
    urgencies = {
        str(v)
        for e in evidence
        for v in (e["result"].get("urgency"), e["result"].get("urgency_ground_truth"))
        if v
    }
    queues = {
        str(v)
        for e in evidence
        for v in (e["result"].get("route_queue"), e["result"].get("route_queue_ground_truth"))
        if v
    }
    if decision.urgency not in urgencies:
        raise ValueError("urgency unsupported by evidence")
    if decision.route_queue not in queues:
        raise ValueError("route_queue unsupported by evidence")


evidence_001 = collect_evidence("CLN-001")
decision_001 = make_model().with_structured_output(TriageDecision).invoke(
    "Return a TriageDecision using only this evidence. Use the CLM claim id as case_id. "
    f"Evidence: {json.dumps(evidence_001)}"
)
validate_grounding(decision_001, evidence_001)

evidence_007 = collect_evidence("CLN-007")
display(
    JSON(
        {
            "CLN-001": {
                "status": "passed",
                "decision": decision_001.model_dump(),
                "tools": [e["tool"] for e in evidence_001],
            },
            "CLN-007": {
                "note": "Incomplete inquiry — no claim number in FNOL; grounding should refuse a CLM case_id.",
                "tools": [e["tool"] for e in evidence_007],
                "has_claim_id": any(
                    e["result"].get("claim_number_ground_truth") for e in evidence_007
                ),
            },
        }
    )
)

<IPython.core.display.JSON object>

## 04 Memory, layered

<img src="Images/d1am_t4_layered_memory.png" width="850" alt="Layered memory: working context, summaries, artifacts, long-term preferences">

- **Working context** — current messages.
- **Summary** — compressed prior turns.
- **Artifact** — durable handoff JSON for another system or day.
- **Preferences** — stable operator/user settings.
- Memory ≠ vector DB. Retrieval stores are one optional backend, not the memory model.
- **Lost-in-the-middle:** keep decision-critical evidence near the active task.

Demos: lost-in-the-middle placement, `InMemorySaver` across two turns, then a small handoff write under `W2D1/outputs/day1/`.

In [13]:
important = "CRITICAL: route_queue for this case must remain fnol_intake."
filler = [f"Routine note {i}: weather delay on unrelated claim." for i in range(24)]
buried = "\n".join(filler[:12] + [important] + filler[12:])
at_end = "\n".join(filler + [important])

probe = make_model(max_tokens=300)
mid = probe.invoke(
    f"From this context, state the CRITICAL route rule in one sentence.\n\n{buried}"
).content
end = probe.invoke(
    f"From this context, state the CRITICAL route rule in one sentence.\n\n{at_end}"
).content
display(JSON({"answer_when_fact_buried": mid, "answer_when_fact_at_end": end}))

checkpointer = InMemorySaver()


def build_checkpointed():
    model = make_model().bind_tools(TOOLS)

    def call_model(state: MessagesState):
        return {"messages": [model.invoke(state["messages"])]}

    def route(state: MessagesState):
        return "tools" if state["messages"][-1].tool_calls else END

    builder = StateGraph(MessagesState)
    builder.add_node("agent", call_model)
    builder.add_node("tools", ToolNode(TOOLS, handle_tool_errors=True))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile(checkpointer=checkpointer)


thread = {"configurable": {"thread_id": "day1-am-cln-001"}, "recursion_limit": TURN_BUDGET}
ckpt_graph = build_checkpointed()
turn1 = ckpt_graph.invoke(
    {
        "messages": [
            SystemMessage(content="Use domain tools for facts. Never invent records."),
            HumanMessage(content="Investigate CLN-001 and summarize evidence."),
        ]
    },
    config=thread,
)
turn2 = ckpt_graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="Using this thread only, state the evidence-backed route_queue. Do not invent."
            )
        ]
    },
    config=thread,
)
snapshot = ckpt_graph.get_state(thread)
summary = (
    "CLN-001 has FNOL + claim + policy evidence; preferred route is fnol_intake."
)
display(
    JSON(
        {
            "thread_id": "day1-am-cln-001",
            "messages_after_turn1": len(turn1["messages"]),
            "messages_after_turn2": len(turn2["messages"]),
            "checkpoint_id": snapshot.config["configurable"].get("checkpoint_id"),
            "working_summary": summary,
            "turn2_answer": turn2["messages"][-1].content,
        }
    )
)

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [14]:
# Explicit learner-run write: small handoff artifact for Day 2 continuity.
HANDOFF_PATH = HANDOFF_DIR / "day1_am_handoff.json"
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

handoff = {
    "schema_version": "1.0",
    "case_id": decision_001.case_id,
    "email_id": "CLN-001",
    "decision": decision_001.model_dump(),
    "evidence": evidence_001,
    "working_summary": summary,
    "preferences": {"operator_style": "concise"},
    "idempotency_key": "day1-am-cln-001-v1",
}
HANDOFF_PATH.write_text(json.dumps(handoff, indent=2), encoding="utf-8")
display(JSON({"wrote": str(HANDOFF_PATH), "artifact": handoff}))

<IPython.core.display.JSON object>

## 05 Agent vs pipeline

<img src="Images/d1am_t5_agent_vs_pipeline.png" width="850" alt="Hardcoded pipeline versus agent tool loop">

Same email, two runtimes: a hardcoded **fnol → claim → policy** pipeline versus the agent tool loop. Compare latency and tool counts, then decide when each wins.

In [15]:
def pipeline_cln001() -> dict:
    started = time.perf_counter()
    fnol = json.loads(fnol_lookup.invoke({"email_id": "CLN-001"}))
    claim = json.loads(claim_lookup.invoke({"claim_id": fnol["claim_number_ground_truth"]}))
    policy = json.loads(policy_lookup.invoke({"policy_id": fnol["policy_id_ground_truth"]}))
    return {
        "latency_seconds": round(time.perf_counter() - started, 3),
        "tool_count": 3,
        "evidence_ids": [fnol["email_id"], claim["claim_id"], policy["policy_id"]],
    }


pipe = pipeline_cln001()

agent_started = time.perf_counter()
agent_out = build_grounded_graph().invoke(
    {
        "messages": [
            SystemMessage(
                content=(
                    "Use tools for all facts. Call fnol_lookup, then claim and policy "
                    "tools for identifiers present in FNOL."
                )
            ),
            HumanMessage(content="Triage CLN-001."),
        ]
    },
    config={"recursion_limit": TURN_BUDGET},
)
agent_tools = [m for m in agent_out["messages"] if isinstance(m, ToolMessage)]
agent_metrics = {
    "latency_seconds": round(time.perf_counter() - agent_started, 3),
    "tool_count": len(agent_tools),
}

display(
    JSON(
        {
            "pipeline": pipe,
            "agent": agent_metrics,
            "when_pipeline_wins": "Known three-step retrieval; lower latency; easier tests.",
            "when_agent_wins": "Next valid action depends on observations; incomplete or branching cases.",
        }
    )
)

<IPython.core.display.JSON object>

## 06 Failure modes & guardrails intro

<img src="Images/d1am_t6_failure_modes_guardrails.png" width="850" alt="Fail-closed grounding and budget stops">

Fail closed when tool evidence is missing. `CLN-001` has complete FNOL/claim/policy evidence; `CLN-007` does not. Also stop on model-call budget / recursion limits and surface `stop_reason` in state.

In [16]:
def try_grounded_decision(email_id: str) -> dict:
    evidence = collect_evidence(email_id)
    try:
        decision = make_model().with_structured_output(TriageDecision).invoke(
            "Return a TriageDecision using only this evidence. "
            "If no claim id exists, still return a structured object but grounding will refuse.\n"
            f"Evidence: {json.dumps(evidence)}"
        )
        validate_grounding(decision, evidence)
        return {"status": "completed", "decision": decision.model_dump(), "evidence_tools": [e["tool"] for e in evidence]}
    except ValueError as err:
        return {
            "status": "refused",
            "refusal_reason": str(err),
            "evidence_tools": [e["tool"] for e in evidence],
        }


complete_case = try_grounded_decision("CLN-001")
incomplete_case = try_grounded_decision("CLN-007")
display(JSON({"CLN-001": complete_case, "CLN-007": incomplete_case}))

<IPython.core.display.JSON object>

In [17]:
class GuardedState(MessagesState):
    model_calls: int
    stop_reason: str


def build_budget_graph(max_model_calls: int):
    model = make_model().bind_tools(TOOLS)

    def call_model(state: GuardedState):
        calls = state.get("model_calls", 0)
        if calls >= max_model_calls:
            return {"stop_reason": "model_call_budget_exhausted"}
        response = model.invoke(state["messages"])
        return {
            "messages": [response],
            "model_calls": calls + 1,
            "stop_reason": "tool_requested" if response.tool_calls else "completed",
        }

    def route(state: GuardedState):
        if state.get("stop_reason") == "model_call_budget_exhausted":
            return END
        return "tools" if state["messages"][-1].tool_calls else END

    builder = StateGraph(GuardedState)
    builder.add_node("agent", call_model)
    builder.add_node("tools", ToolNode(TOOLS, handle_tool_errors=True))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()


# Force a budget stop: allow only 1 model call while the prompt asks for tools.
budget_result = build_budget_graph(max_model_calls=1).invoke(
    {
        "messages": [
            SystemMessage(
                content=(
                    "Always call fnol_lookup, claim_lookup, and policy_lookup before answering. "
                    "Keep requesting tools until all three exist."
                )
            ),
            HumanMessage(content="Fully investigate CLN-001."),
        ],
        "model_calls": 0,
        "stop_reason": "not_started",
    },
    config={"recursion_limit": 4},
)

display(
    JSON(
        {
            "model_calls": budget_result["model_calls"],
            "stop_reason": budget_result["stop_reason"],
            "recursion_limit": 4,
            "message_count": len(budget_result["messages"]),
        }
    )
)

<IPython.core.display.JSON object>

## Exercises

Complete these after running the sections above. Edit copies of the earlier patterns in the workspace cells — do **not** paste solution code here. Solutions (for instructors / self-check) live in [`_EXERCISES_SOLUTIONS.ipynb`](_EXERCISES_SOLUTIONS.ipynb).

### Exercise AM-1 — Force a missing FNOL lookup

**Concept:** tool contracts return authoritative records; `found: false` is a valid observation, not an excuse to invent.

**Task:** Change the forced-tool demonstration so it asks for **CLN-007** instead of CLN-001.

**Expected different result:** the tool result shows the email was not found (or has no usable claim/policy IDs). You must **not** build a completed grounded `TriageDecision` from empty evidence.

**TODO hints (no code):**
- TODO: find the forced `bind_tools` / `tool_choice` invoke that currently targets CLN-001
- TODO: change only the operator request identifier
- TODO: print the raw tool JSON and assert it does not contain a real claim id
- TODO: optionally call your grounding helper and show `status: refused` (or equivalent)

In [ ]:
# Exercise AM-1 workspace
# TODO: reuse the forced-tool pattern from §03 with email_id CLN-007
# TODO: display the tool_result and confirm it is not a found FNOL row with claim/policy IDs
# TODO: (optional) run grounding and show a refusal for incomplete evidence

raise NotImplementedError("Exercise AM-1: implement the CLN-007 forced lookup yourself")

### Exercise AM-2 — Relax the model-call budget

**Concept:** application-owned stop conditions (budgets) change observable `stop_reason` and how far the agent loop proceeds.

**Task:** Rebuild the guarded budget graph from §06, but allow **3** model calls instead of **1**, with the same “call all three tools” system prompt for CLN-001.

**Expected different result:** you should **not** stop primarily for `model_call_budget_exhausted` after a single model turn. Compare `model_calls`, `stop_reason`, and whether tool messages appear.

**TODO hints (no code):**
- TODO: copy `build_budget_graph` usage from the failure-modes cell
- TODO: change only the budget argument (1 → 3)
- TODO: keep recursion_limit high enough that the budget—not recursion—is the interesting control
- TODO: display a small JSON diff: `{budget_1: ..., budget_3: ...}`

In [ ]:
# Exercise AM-2 workspace
# TODO: invoke build_budget_graph(max_model_calls=3) with the same CLN-001 investigate prompt
# TODO: compare against the earlier max_model_calls=1 run (rerun that cell if needed)
# TODO: show model_calls and stop_reason for both budgets

raise NotImplementedError("Exercise AM-2: compare budget=1 vs budget=3 yourself")

### Operator checklist

- Treat model text as a proposal; treat tool rows as facts.
- Persist evidence, decisions, and `stop_reason`—not hidden reasoning.
- Prefer a pipeline for known paths; use an agent only when runtime choice adds value.
- Fail closed on missing claim/policy evidence (`CLN-007` pattern).